### Using the HadISD datset (version 3.4.0.2023f) with PyEarthTools
HadISD is a global sub-daily dataset based on the ISD dataset from NOAA's NCEI. As well as station selection criteria, a suite of quality control tests has been run on the major climatological variables.

The dataset can be downloaded here: https://www.metoffice.gov.uk/hadobs/hadisd/v340_2023f/download.html

In [ ]:
import datetime
import numpy as np
from pathlib import Path


import pyearthtools.pipeline as petpipe
import pyearthtools.data as petdata
from pyearthtools.tutorial.HadisdDataClass import HadISDIndex

# Create an ordered list of all available HadISD station IDs
The  HadISDIndex class contains a method called "get_alL station_ids" that can be used to create a list of all available station IDS. This list can be used to select, for example, the first_ten stations from all stations. It can then be passed used to select data from within the pipeline. It's also possible to pass a single station ID as a string to retrieve data for a single station.

TODO:
- It would be good if a range of stations can be provided, or a boundary with stations inside

In [ ]:
hadisd = HadISDIndex()
all_stations = hadisd.get_all_station_ids(Path("/Users/joelmiller/Projects/data/hadisd"))
all_stations_ordered = sorted(all_stations)
print(f"Total number of stations: {len(all_stations_ordered)}")
first_ten_stations = all_stations_ordered[:10]
print(first_ten_stations)

In [ ]:
varname_val_map = {
        "total_cloud_cover": -999., 
        "low_cloud_cover": -999., 
        "mid_cloud_cover": -999.,
        "high_cloud_cover": -999.
    }

flagged_labels = [
        'temperatures', 'dewpoints', 'slp',
        'stnlp', 'windspeeds', 'winddirs', 
        'total_cloud_cover', 'low_cloud_cover', 'mid_cloud_cover', 
        'high_cloud_cover', 'precip1_depth', 'precip2_depth', 
        'precip3_depth', 'precip6_depth', 'precip9_depth',
        'precip12_depth', 'precip15_depth', 'precip18_depth', 
        'precip24_depth'
    ]

# Pipeline Definition

In [ ]:
# Custom operation to remove redundent coordinates
# TODO: Find a home for this in pyearthtools and refine logic
class SqueezeStationCoordinates(petpipe.Operation):
    def __init__(self, coords=("latitude", "longitude", "elevation")):
        super().__init__()
        self.coords = coords
    def apply_func(self, ds):
        for coord in self.coords:
            if coord in ds.coords and len(ds[coord].shape) > 1:
                ds[coord] = ds[coord].squeeze()
        return ds
    
    # Undo function added otherwise pyearthtools will complain
    def undo_func(self, ds):
        # No undo operation needed for this operation
        return ds

In [ ]:
# Custom operation split data into feature and target variables
# TODO: Find a home for this in pyearthtools and refine logic
class FeatureTargetSplit(petpipe.Operation):
    """
    Split dataset into features (X) and target (y) for ML.
    """
    def __init__(self, target_vars, drop_target_from_x=True):
        super().__init__()
        self.target_vars = [target_vars] if isinstance(target_vars, str) else list(target_vars)
        self.drop_target_from_x = drop_target_from_x

    def apply_func(self, ds):
        # X: all variables except target(s)
        if self.drop_target_from_x:
            feature_vars = [v for v in ds.data_vars if v not in self.target_vars]
        else:
            feature_vars = list(ds.data_vars)
        X = ds[feature_vars]
        # y: just the target(s)
        y = ds[self.target_vars]
        return (X, y)
    
    # Undo function added otherwise pyearthtools will complain
    def undo_func(self, ds):
        return ds

In [ ]:
# Hadisd accepts "all" as an argument to get all stations.
# A single station can also be specified, e.g. "010010-99999".
# Or, a list of station IDs can be passed.

# TODO: Make numpy conversion step work.

data_prep_pipe = petpipe.Pipeline(
    petdata.archive.hadisd(station = first_ten_stations), #, variables = ["total_cloud_cover", "temperatures", "flagged_obs"]), # Commented out variable selection whilst looking into bug in BaseTransforms
    SqueezeStationCoordinates(), # Squeeze coordinates to 1D if they are 2D: Remove redundant dimensions
    petdata.transforms.variables.Drop("reporting_stats"),
    petdata.transforms.values.SetMissingToNaN(varname_val_map),
    petdata.transforms.values.AddFlaggedObs(flagged_labels),

    #petdata.transforms.variables.Select(variables=["total_cloud_cover", "temperatures", "dewpoints", "flagged_obs"]), # Use transform in pipeline to bypass bug in BaseTransforms
    FeatureTargetSplit(target_vars="flagged_obs"),
    #petpipe.operations.xarray.conversion.ToNumpy(), # Doesn't work for HadISD in current format. Must fix
    #petdata.transofmrs.derive.WetBulbTemp(), # Just an idea for now. Should probably be in a separate pipeline where heavier processing is done
)
data_prep_pipe

In [ ]:
x, y = data_prep_pipe["1970-01-01T07"]
x

In [ ]:
y

## Impute Median Values to get clearer picture of data structure where NaNs present

Everything here is for my understanding

In [ ]:
ds = x

def median_impute_per_station(ds):
    ds_imputed = ds.copy()
    for var in ds.data_vars:
        arr = ds[var].values  # shape: (station, time)
        # Compute median for each station (axis=1 is time)
        medians = np.nanmedian(arr, axis=1)
        # Broadcast medians to the shape of arr
        arr_imputed = np.where(np.isnan(arr), medians[:, None], arr)
        ds_imputed[var].values = arr_imputed
    return ds_imputed

# Usage:
x_imputed = median_impute_per_station(x)

In [ ]:
ds = y

def median_impute_per_station(ds):
    ds_imputed = ds.copy()
    for var in ds.data_vars:
        arr = ds[var].values  # shape: (station, time)
        # Compute median for each station (axis=1 is time)
        medians = np.nanmedian(arr, axis=1)
        # Broadcast medians to the shape of arr
        arr_imputed = np.where(np.isnan(arr), medians[:, None], arr)
        ds_imputed[var].values = arr_imputed
    return ds_imputed

# Usage:
y_imputed = median_impute_per_station(y)

In [ ]:
x_imputed

In [ ]:
y_imputed

In [ ]:
#view flagged obs for station 0 and time 0 (These are imputed values, otherwise they would be NaN and give no clues)
station_index = 4
time_index = 100000
flagged_obs = y_imputed["flagged_obs"].values[station_index, time_index]
print(f"Flagged observations for station {station_index} at time {time_index}: {flagged_obs}")

In [ ]:
# view qc data for station 0 and time 0
station_index = 0
time_index = 0 
qc_data = x["quality_control_flags"].values[station_index, time_index]
print (f"test for station {station_index} at time {time_index}: {qc_data}")


In [ ]:
qc = x["quality_control_flags"].values
# Find indices where qc is not 0 and not NaN
indices = np.where((qc != 0) & ~np.isnan(qc))

print(f"Number of non-zero quality control flags: {len(indices[0])}")
print("Indices (station, time, test):")
for idx in zip(*indices):
    print(idx)


In [ ]:
print(x["quality_control_flags"].coords["test"].values)

In [ ]:
print(y["flagged_obs"].coords["flagged"].values)

# Messing With XGBoost

In [ ]:
from xgboost import XGBClassifier
# read data
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
# data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(x['data'], y['target'], test_size=.2)
X_train

In [ ]:
# create model instance
bst = XGBClassifier(n_estimators=2, max_depth=2, learning_rate=1, objective='binary:logistic')
# fit model
bst.fit(X_train, y_train)
# make predictions
preds = bst.predict(X_test)

In [ ]:
# Set up the caching mechanism to store processed data in the specified folder with .npy extension
cache_folder = Path('/Users/joelmiller/Desktop/cache')
cache_folder.mkdir(parents=True, exist_ok=True)

caching_step = petpipe.modifications.Cache(
    cache_folder, 
    pattern_kwargs={'extension': 'nc'},
    # cache_validity='delete'
)

In [ ]:
# Add the caching step to the pipeline
data_preparation_normed = petpipe.Pipeline(
    data_prep_pipe,
    caching_step
)

In [ ]:
x, y = data_preparation_normed["1970-01-01T06"]
x

In [ ]:
# open single netcdf file with xarray
import xarray as xr
ds = xr.open_dataset("/Users/joelmiller/Desktop/cache/1970/01/01/19700101T0600.nc/0.nc")
ds

In [ ]:
for coord in ["latitude", "longitude", "elevation"]:
    if coord in ds:
        ds = ds.set_coords(coord)
ds

In [ ]:
for coord in ["latitude", "longitude", "elevation"]:
    if coord in ds.coords and "coordinate_length" in ds[coord].dims:
        ds[coord] = ds[coord].squeeze("coordinate_length")

ds

In [ ]:
ds_numpy= ds.pipe(petpipe.operations.xarray.conversion.ToNumpy())
ds_numpy

In [ ]:
for coord in ["latitude", "longitude", "elevation"]:
    if coord in ds:
        ds = ds.set_coords(coord)

ds

In [ ]:
import pyearthtools.data as petdata
# select only total_cloud_cover
ds_all_tcc = ds.pipe(petdata.transforms.variables.Select("total_cloud_cover", "dewpoints", "temperatures"))
ds_all_tcc

In [ ]:
print(ds_all_tcc)
for name, coord in ds_all_tcc.coords.items():
    print(f"{name}: {coord.shape}")
for name, var in ds_all_tcc.data_vars.items():
    print(f"{name}: {var.shape}")

In [ ]:
ds_numpy= ds_all_tcc.pipe(petpipe.operations.xarray.conversion.ToNumpy())
ds_numpy

In [ ]:
# Show the lat and lon coordinatesfor the first station
print(f"Latitude: {ds.latitude.values[1]}")
print(f"Longitude: {ds.longitude.values[1]}")




# Notes:
- Dropping variables results in coordinates being dropped, which is not what we want.
- Calculating wetbulb temps for just 10 stations takes a very long time; therefore, 
  I wonder if it is best suited to a separate pre-processing pipeline to get the data into a more suitable format ready for manipulations to be done.
- Don't know what combination of stations you want when selecting nearest neighbours, so want to cahce on a per-station basis. This might be differnt with zarr.

TODO:
- Change preprocessing so that coordinates are not dropped when dropping or selecting variables.

In some cases you will get the following error when passing a date time to a pipeline object: `IndexWarning: Could not find time in dataset to select on. Petdt('1931-01-01T07')`<br>

This indicates that data for the datetime you chose does not exist. In this case PET will load all data from your station selection.

In [ ]:
# show total_cloud_cover from ds
tcc = ds["total_cloud_cover"]
tcc

# For specific Stations

In [ ]:
from pyearthtools.tutorial.HadisdDataClass import HadISDIndex
from pathlib import Path
hadisd = HadISDIndex(["010014-99999", "010010-99999", "010030-99999"], variables=["total_cloud_cover", "temperatures", "dewpoints", "flagged_obs"])
paths = hadisd.filesystem()
paths


In [ ]:
# Check waht class attribute is set for the HadISDIndex object
print(hadisd.station)
print(hadisd.variables)

In [ ]:
file_paths = list(paths.values())
file_paths

In [ ]:
ds = hadisd.load(file_paths)
ds

In [ ]:
lat_values = ds["latitude"].values
lon_values = ds["longitude"].values
elev_values = ds["elevation"].values
lat_values

import numpy as np

# Ensure 1D arrays for each coordinate
lat_values = np.asarray(ds["latitude"].values).reshape(-1)
lon_values = np.asarray(ds["longitude"].values).reshape(-1)
elev_values = np.asarray(ds["elevation"].values).reshape(-1)

In [ ]:
ds = ds.assign_coords(
    latitude=("station", lat_values),
    longitude=("station", lon_values),
    elevation=("station", elev_values),
)

In [ ]:
# set all missing values to NaN
ds_3_nan = ds.pipe(petdata.transforms.values.SetMissingToNaN(varname_val_map))
ds_3_nan

In [ ]:
# View all lat/lon values
ds_3_nan["latitude"].values, ds_3_nan["longitude"].values

In [ ]:
import matplotlib.pyplot as plt
flagged_counts = np.sum(~np.isnan(ds['flagged_obs'].values), axis=(1,2))  # sum over time and flagged
plt.bar(range(ds.dims['station']), flagged_counts)
plt.xlabel('Station')
plt.ylabel('Number of flagged obs')
plt.show()

In [ ]:
import pandas as pd
time = pd.to_datetime(ds['time'].values)
flag = ~np.isnan(ds['flagged_obs'].isel(station=0, flagged=0).values)
flag_series = pd.Series(flag, index=time)
flag_counts = flag_series.resample('y').sum()
flag_counts.plot()
plt.ylabel('Flagged obs per year')
plt.show()

# For all stations

In [ ]:
hadisd = HadISDIndex("all")

all_station_ids = hadisd.get_all_station_ids(Path("/Users/joelmiller/Projects/data/hadisd"))
print(all_station_ids[:10])
print(len(all_station_ids))


In [ ]:
hadisd = HadISDIndex(all_station_ids)
paths = hadisd.filesystem() 
paths = list(paths.values())

In [ ]:
ds_all = hadisd.load(paths)
ds_all

In [ ]:
import pyearthtools.data as petdata
# select only total_cloud_cover
ds_all_tcc = ds_all.pipe(petdata.transforms.variables.Select("total_cloud_cover"))
ds_all_tcc


In [ ]:
# Manuall apply the transform
varname_val_map = {
        "total_cloud_cover": -999., 
        "low_cloud_cover": -999., 
        "mid_cloud_cover": -999.,
        "high_cloud_cover": -999.
    }

ds_all_tcc_nan = ds_all.pipe(petdata.transforms.values.SetMissingToNaN(varname_val_map))
ds_all_tcc_nan

In [ ]:
tcc = ds_all["total_cloud_cover"]
tcc

# Stuff in this section is not used in the tutorial, but is useful for testing and exploration

In [ ]:

# I think used for preprocessing to zarr?
date_range=(datetime.datetime(1986,11,20,12,0), datetime.datetime(1986,11,21,0,0))

In [ ]:
varname_val_map = {
        "total_cloud_cover": -999., 
        "low_cloud_cover": -999., 
        "mid_cloud_cover": -999.,
        "high_cloud_cover": -999.
    }

# Probably sensible to accetp a list since some variables might have multiple values
# This would look like:
# varname_val_map = {
#         "total_cloud_cover": [-888., -999.],
#         "low_cloud_cover": [-888., -999.],
#         "mid_cloud_cover": [-999.],
#         "high_cloud_cover": [-999.]
#     }

In [ ]:
# train/validation/test split dates
train_start = "1970-01-01T00"
train_end = "2022-12-31T23"

In [ ]:
data_prep_pipe = petpipe.Pipeline(
    # petdata.archive.hadisd(("010010-99999"), variables = ["total_cloud_cover", "temperatures", "flagged_obs"]),
    # petdata.archive.hadisd(station = ["010014-99999"], variables = ["total_cloud_cover", "temperatures", "flagged_obs"]),
    
    # Current options for indexing
    #petdata.archive.hadisd(["010014-99999", "010010-99999", "010030-99999"]),    
    petdata.archive.hadisd("010010-99999", variables=["total_cloud_cover", "temperatures", "flagged_obs"]),

    petdata.transforms.values.AddFlaggedObs(flagged_labels),
    petdata.transforms.values.SetMissingToNaN(varname_val_map),
    
    # Possible ways of indexing in the future:
    # petdata.archive.hadisd(country_code = "24", station_id = "010010", nearest_neighbors = 12, lon= 0.0, lat = 0.0), # Geospatial fence 
    # petdata.archive.hadisd("010014-99999"), # Geospatial fence
)
data_prep_pipe

# Don't know what combination of stations you want when selecting nearest neighbours, so want to cahce on a per-station basis

# Use the section below to build tests to check that all pipeline steps are working as expected

## Test SetMissingToNaN

In [ ]:
import xarray as xr
from pyearthtools.data.transforms.values import SetMissingToNaN
import numpy as np

def test_set_missing_to_nan():
    data = xr.Dataset({
        "total_cloud_cover": ("time", [0, -999, 50]),
        "low_cloud_cover": ("time", [10, -999, 20]),
    })

    varname_val_map = {
        "total_cloud_cover": -99.0,
        "low_cloud_cover": -999.0,
    }

    transform = SetMissingToNaN(varname_val_map)
    transformed_data = transform.apply(data)

    if np.isnan(transformed_data["total_cloud_cover"].data[1]):
        print("Total cloud cover at index 1 is NaN")
    else:
        print("Total cloud cover at index 1 is not NaN")
    
    if np.isnan(transformed_data["low_cloud_cover"].data[1]):
        print("Low cloud cover at index 1 is NaN")
    
    if transformed_data["total_cloud_cover"].data[2] == 50:
        print("Total cloud cover at index 2 is 50")

In [ ]:
test_set_missing_to_nan()

## Test AddFlaggedObs

In [ ]:
import xarray as xr
import numpy as np
from pyearthtools.data.transforms.values import AddFlaggedObs

def test_add_flagged_obs():
    # Mock dataset with flagged observations
    data = xr.Dataset(
        {
            "temperatures": (("time",), [np.nan, 15.0, np.nan]),
            "dewpoints": (("time",), [np.nan, 10.0, np.nan]),
            "flagged_obs": (
                ("time", "flagged"),
                [
                    [20.0, np.nan],  # Flagged data for time=0
                    [np.nan, np.nan],  # No flagged data for time=1
                    [25.0, 12.0],  # Flagged data for time=2
                ],
            ),
        },
        coords={
            "time": [0, 1, 2],
            "flagged": [0, 1],  # Indices for flagged variables
        },
    )

    # Add flagged_value attribute to variables
    data["temperatures"].attrs["flagged_value"] = -999.0
    data["dewpoints"].attrs["flagged_value"] = -999.0

    # Define flagged labels corresponding to the flagged dimension
    flagged_labels = ["temperatures", "dewpoints"]

    # Apply the AddFlaggedObs transform
    transform = AddFlaggedObs(flagged_labels)
    transformed_data = transform.apply(data)

    # Assert that flagged data has been restored
    assert np.allclose(
        transformed_data["temperatures"].data, [20.0, 15.0, 25.0], equal_nan=True
    )
    assert np.allclose(
        transformed_data["dewpoints"].data, [np.nan, 10.0, 12.0], equal_nan=True
    )

    print("Test passed: Flagged observations were correctly restored.")

# Run the test
test_add_flagged_obs()